In [5]:
from bs4 import BeautifulSoup, Comment
import requests
import pandas as pd

In [3]:
def extract_from_commented_html(soup: BeautifulSoup, id_contains: str) -> BeautifulSoup:
    """
    Szuka komentarzy w HTML zawierających dane, np. <div> zakomentowany w <!-- ... -->
    i zwraca zawartość jako parsowalny fragment BeautifulSoup.
    
    :param soup: obiekt BeautifulSoup z requests.get(url).text
    :param id_contains: fragment tekstu identyfikujący interesujący blok np. "players_of_the_week"
    :return: BeautifulSoup z zawartością z komentarza (lub None jeśli nie znaleziono)
    """
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        if id_contains in comment:
            return BeautifulSoup(comment, "html.parser")
    
    print(f"❌ Nie znaleziono komentarza zawierającego '{id_contains}'")
    return None

In [ ]:
url = "https://www.basketball-reference.com/leagues/NBA_2020.html"
res = requests.get(url)
soup = BeautifulSoup(res.text, "html.parser")

section = extract_from_commented_html(soup, "players_of_the_week_and_month")
if section:
    print(section.find_all("h3")[:])


[<h3>2019-20</h3>, <h3> Players of the Week</h3>, <h3> Players of the Week</h3>, <h3> Players of the Month</h3>, <h3> Rookies of the Month</h3>, <h3> Coaches of the Month</h3>, <h3> Players of the Week</h3>, <h3> Players of the Month</h3>, <h3> Rookies of the Month</h3>, <h3> Coaches of the Month</h3>]


In [80]:
def get_award_counts(season):

    season_int = int(season)
    season_label = f"{season_int - 1}-{str(season_int)[-2:]}"  # np. 2019-20

    url = "https://www.basketball-reference.com/leagues/NBA_2020.html"
    res = requests.get(url)
    res.encoding = "utf-8"

    soup = BeautifulSoup(res.text, "html.parser")

    section = extract_from_commented_html(soup, "players_of_the_week_and_month")
    if section is None:
        print(f"Nie znaleziono sekcji 'players_of_the_week_and_month' w sezonie {season_label}.")
        return pd.DataFrame()

    potw, potm, rotm = [], [], []
    current_award = ""

    for tag in section.find_all(["h3", "a"]):
        #print(tag)
        text = tag.get_text().lower()
        #print(f"TAG NAME: {tag.name}")
        #print(f"TEXT: {text}")
        if tag.name == "h3":
            if "players of the week" in text:
                current_award = "potw"
            elif "players of the month" in text and "rookie" not in text:
                current_award = "potm"
            elif "rookies of the month" in text:
                current_award = "rotm"
            else:
                current_award = ""

        elif tag.name == "a":
            player = text.title()
            if current_award == "potw":
                potw.append(player)
            elif current_award == "potm":
                potm.append(player)
            elif current_award == "rotm":
                rotm.append(player)

    print(f"Znaleziono {len(potw)} graczy tygodnia, {len(potm)} graczy miesiąca i {len(rotm)} debiutantów miesiąca w sezonie {season_label}.")
    df = pd.DataFrame()
    df["Player"] = list(set(potw + potm + rotm))
    df["potw_count"] = df["Player"].apply(lambda x: potw.count(x))
    df["potm_count"] = df["Player"].apply(lambda x: potm.count(x))
    df["rookie_of_month_count"] = df["Player"].apply(lambda x: rotm.count(x))

    return df


In [81]:
df_awards = get_award_counts("2020")
df_awards.iloc[:].sort_values("potw_count", ascending=False)

Znaleziono 36 graczy tygodnia, 8 graczy miesiąca i 8 debiutantów miesiąca w sezonie 2019-20.


,Player,potw_count,potm_count,rookie_of_month_count
28,Giannis Antetokounmpo,4,3,0
6,Lebron James,3,2,0
13,Pascal Siakam,2,0,0
24,Jaylen Brown,2,0,0
15,James Harden,2,1,0
5,Anthony Davis,2,0,0
18,Damian Lillard,2,0,0
4,Jayson Tatum,1,1,0
9,Josh Richardson,1,0,0
7,Spencer Dinwiddie,1,0,0


In [3]:
def get_all_nba_team(season="2020"):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}.html"
    res = requests.get(url)
    res.encoding = "utf-8"  # bardzo ważne dla znaków specjalnych!
    soup = BeautifulSoup(res.text, "html.parser")

    # Szukamy komentarzy zawierających dane All-NBA
    comment_html = None
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        if "div_all-nba" in comment:
            comment_html = comment
            break

    if comment_html is None:
        print("❌ Nie znaleziono danych All-NBA.")
        return pd.DataFrame()

    section = BeautifulSoup(comment_html, "html.parser")

    all_teams = []

    for team_id, team_num in zip(["all-nba_1", "all-nba_2", "all-nba_3"], [1, 2, 3]):
        team_div = section.find("div", id=team_id)
        if team_div:
            players = [a.text.strip() for a in team_div.find_all("a")]
            for player in players:
                all_teams.append({"Player": player, "all_nba_team": team_num})

    return pd.DataFrame(all_teams)


In [6]:
df_all_nba = get_all_nba_team("2020")
df_all_nba.head(15)


,Player,all_nba_team
0,Giannis Antetokounmpo,1
1,Anthony Davis,1
2,Luka Dončić,1
3,James Harden,1
4,LeBron James,1
5,Nikola Jokić,2
6,Kawhi Leonard,2
7,Damian Lillard,2
8,Chris Paul,2
9,Pascal Siakam,2


In [7]:
def get_all_rookie_team(season="2020"):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}.html"
    res = requests.get(url)
    res.encoding = "utf-8"  # bardzo ważne dla znaków specjalnych!
    soup = BeautifulSoup(res.text, "html.parser")
    

    # Szukamy komentarzy zawierających dane All-Rookie
    comment_html = None
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        if "div_all-rookie" in comment:
            comment_html = comment
            break

    if comment_html is None:
        print("❌ Nie znaleziono danych All-Rookie.")
        return pd.DataFrame()

    section = BeautifulSoup(comment_html, "html.parser")

    all_teams = []

    for team_id, team_num in zip(["all-rookie_1", "all-rookie_2"], [1, 2]):
        team_div = section.find("div", id=team_id)
        if team_div:
            players = [a.text.strip() for a in team_div.find_all("a")]
            for player in players:
                all_teams.append({"Player": player, "all_rookie_team": team_num})

    return pd.DataFrame(all_teams)


In [8]:
df_all_rookie = get_all_rookie_team("2020")
df_all_rookie.head(10)


,Player,all_rookie_team
0,Brandon Clarke,1
1,Ja Morant,1
2,Kendrick Nunn,1
3,Eric Paschall,1
4,Zion Williamson,1
5,Terence Davis,2
6,Rui Hachimura,2
7,Tyler Herro,2
8,P.J. Washington,2
9,Coby White,2


In [ ]:
def build_awards_column(df_all_nba, df_all_rookie):
    """Tworzy jedną kolumnę 'awards' dla klasyfikacji All-NBA i All-Rookie"""
    awards = {}

    # All-NBA Teams
    for _, row in df_all_nba.iterrows():
        player = row["Player"].strip()
        team = row["all_nba_team"]
        if player:  
            awards[player] = team  # 1, 2, 3

    # All-Rookie Teams
    for _, row in df_all_rookie.iterrows():
        player = row["Player"].strip()
        team = row["all_rookie_team"]
        if player:
            # Jeśli zawodnik już ma All-NBA, zostaw tamto
            if player not in awards:
                awards[player] = team + 3  # 4, 5

    # Zwróć jako DataFrame
    df_awards = pd.DataFrame.from_dict(awards, orient="index", columns=["awards"]).reset_index()
    df_awards = df_awards.rename(columns={"index": "Player"})
    
    return df_awards


In [12]:
df_awards = build_awards_column(df_all_nba, df_all_rookie)
print(df_awards.head(25))


                   Player  awards
0   Giannis Antetokounmpo       1
1           Anthony Davis       1
2             Luka Dončić       1
3            James Harden       1
4            LeBron James       1
5            Nikola Jokić       2
6           Kawhi Leonard       2
7          Damian Lillard       2
8              Chris Paul       2
9           Pascal Siakam       2
10           Jimmy Butler       3
11            Rudy Gobert       3
12            Ben Simmons       3
13           Jayson Tatum       3
14      Russell Westbrook       3
15         Brandon Clarke       4
16              Ja Morant       4
17          Kendrick Nunn       4
18          Eric Paschall       4
19        Zion Williamson       4
20          Terence Davis       5
21          Rui Hachimura       5
22            Tyler Herro       5
23        P.J. Washington       5
24             Coby White       5
